# Interactive Text Generation Demo

This notebook mirrors the Streamlit demo app (`src/demo/app.py`) logic in a
notebook-friendly format. It shows how to:

1. Scan for and load a model checkpoint
2. Create a `TextGenerator`
3. Generate text with configurable parameters
4. Render Arabic text with RTL support
5. Measure generation time

> **Note:** The Streamlit app provides a richer UI experience. Run it with:
> ```bash
> cd SDAIA-gpt-project2026
> streamlit run src/demo/app.py
> ```

---
## 1. Imports and Setup

In [ ]:
from __future__ import annotations

import glob
import os
import re
import sys
import time

import torch

# Allow imports from the project root
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from src.config import GPTConfig
from src.generation.generator import TextGenerator
from src.model.transformer import GPTModel
from src.tokenizer.bpe_tokenizer import BPETokenizer

PROJECT_ROOT = os.path.abspath('..')
PRETRAINED_DIR = os.path.join(PROJECT_ROOT, 'checkpoints', 'pretrained')
FINETUNED_DIR = os.path.join(PROJECT_ROOT, 'checkpoints', 'finetuned')
print('Project root:', PROJECT_ROOT)

---
## 2. Checkpoint Scanner

We scan `checkpoints/pretrained/` and `checkpoints/finetuned/` for `.pt` files.
This mirrors the `_scan_checkpoints()` helper in the Streamlit app.

In [ ]:
def scan_checkpoints():
    """Return a dict of display_name -> checkpoint_path."""
    checkpoints = {}
    for ckpt in sorted(glob.glob(os.path.join(PRETRAINED_DIR, '*.pt'))):
        checkpoints[f'pretrained / {os.path.basename(ckpt)}'] = ckpt
    for ckpt in sorted(glob.glob(os.path.join(FINETUNED_DIR, '*.pt'))):
        checkpoints[f'finetuned / {os.path.basename(ckpt)}'] = ckpt
    return checkpoints

available = scan_checkpoints()
if available:
    print(f'Found {len(available)} checkpoint(s):')
    for name in available:
        print(f'  - {name}')
else:
    print('No checkpoints found. We will create a tiny model for demo purposes.')

---
## 3. Load Model from Checkpoint

Each checkpoint stores `model_state_dict` and `config` (a `GPTConfig` instance
or dict). We reconstruct the model and load the weights.

If no checkpoint is available, we create a tiny untrained model so the rest of
the notebook still runs.

In [ ]:
def find_tokenizer_path():
    """Look for a saved tokenizer JSON in common locations."""
    candidates = [
        os.path.join(PROJECT_ROOT, 'tokenizer.json'),
        os.path.join(PROJECT_ROOT, 'data', 'tokenizer.json'),
        os.path.join(PROJECT_ROOT, 'checkpoints', 'tokenizer.json'),
    ]
    for p in candidates:
        if os.path.isfile(p):
            return p
    for p in glob.glob(os.path.join(PROJECT_ROOT, 'checkpoints', '**', 'tokenizer.json'), recursive=True):
        return p
    return None


def load_model_and_tokenizer(ckpt_path):
    """Load a checkpoint and return (model, tokenizer)."""
    checkpoint = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    cfg = checkpoint.get('config')
    if isinstance(cfg, dict):
        config = GPTConfig(**cfg)
    elif isinstance(cfg, GPTConfig):
        config = cfg
    else:
        config = GPTConfig()

    model = GPTModel(config)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    tokenizer = BPETokenizer(vocab_size=config.vocab_size)
    tok_path = find_tokenizer_path()
    if tok_path:
        tokenizer.load(tok_path)
        print(f'Loaded tokenizer from {tok_path}')
    else:
        tokenizer.train('hello world مرحبا بالعالم')
        print('No saved tokenizer found — trained a minimal one.')

    print(f'Model: {model.count_parameters():,} parameters')
    return model, tokenizer


# Load from checkpoint or create a tiny demo model
if available:
    ckpt_name = list(available.keys())[0]
    print(f'Loading: {ckpt_name}')
    model, tokenizer = load_model_and_tokenizer(available[ckpt_name])
else:
    print('Creating a tiny untrained model for demonstration...')
    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train('the cat sat on the mat ' * 20 + 'مرحبا بالعالم ' * 10)
    config = GPTConfig(
        vocab_size=len(tokenizer.vocab), d_model=64, num_heads=4,
        num_layers=2, max_seq_len=128, dropout_rate=0.0)
    model = GPTModel(config)
    model.eval()
    print(f'Tiny model: {model.count_parameters():,} parameters')

---
## 4. Arabic RTL Text Rendering

The Streamlit app uses HTML/CSS injection to render Arabic text right-to-left.
In a notebook we can use `IPython.display.HTML` for the same effect.

In [ ]:
from IPython.display import HTML, display


def contains_arabic(text: str) -> bool:
    """Return True if text contains Arabic script characters."""
    return bool(re.search(
        r'[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF'
        r'\uFB50-\uFDFF\uFE70-\uFEFF]', text))


def render_text(text: str) -> None:
    """Display text with RTL support when Arabic is detected."""
    if contains_arabic(text):
        escaped = (text.replace('&', '&amp;')
                       .replace('<', '&lt;')
                       .replace('>', '&gt;'))
        display(HTML(
            f'<div dir="rtl" style="text-align:right; '
            f'font-size:1.1em; line-height:1.8; '
            f'padding:10px; background:#f8f8f8; '
            f'border-radius:6px;">{escaped}</div>'))
    else:
        print(text)


# Quick test
render_text('Hello, world!')
render_text('مرحبا بالعالم')

---
## 5. Text Generation Helper

This wraps `TextGenerator.generate()` with timing and error handling,
mirroring the Streamlit app's generate button logic.

In [ ]:
GENERATION_TIMEOUT = 60  # seconds


def generate_text(
    generator: TextGenerator,
    prompt: str,
    max_new_tokens: int = 100,
    temperature: float = 1.0,
    top_k: int | None = None,
    top_p: float | None = None,
) -> tuple[str, list[int], float]:
    """Generate text and return (text, token_ids, elapsed_seconds)."""
    if not prompt or not prompt.strip():
        raise ValueError('Prompt must be non-empty.')

    start = time.time()
    text, ids = generator.generate(
        prompt=prompt.strip(),
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
    )
    elapsed = time.time() - start

    if elapsed > GENERATION_TIMEOUT:
        print(f'WARNING: Generation took {elapsed:.1f}s '
              f'(exceeds {GENERATION_TIMEOUT}s timeout threshold)')

    return text, ids, elapsed


generator = TextGenerator(model, tokenizer)
print('Generator ready.')

---
## 6. Generation Examples

### 6.1 Greedy Decoding (`temperature=0`)

Deterministic — always picks the highest-probability token.

In [ ]:
prompt = 'the cat'
text, ids, elapsed = generate_text(generator, prompt, max_new_tokens=30, temperature=0)
print(f'Prompt:  {prompt!r}')
print(f'Output:  {text!r}')
print(f'Tokens:  {len(ids)} in {elapsed:.2f}s')
render_text(text)

### 6.2 Temperature Sampling

Higher temperature → more randomness.

In [ ]:
text, ids, elapsed = generate_text(
    generator, 'the cat', max_new_tokens=30, temperature=0.8)
print(f'Temperature=0.8: {text!r}  ({len(ids)} tokens, {elapsed:.2f}s)')

### 6.3 Top-k Sampling

Only the top-k most probable tokens are considered.

In [ ]:
text, ids, elapsed = generate_text(
    generator, 'the cat', max_new_tokens=30, temperature=1.0, top_k=5)
print(f'Top-k=5: {text!r}  ({len(ids)} tokens, {elapsed:.2f}s)')

### 6.4 Top-p (Nucleus) Sampling

Keeps the smallest set of tokens whose cumulative probability exceeds *p*.

In [ ]:
text, ids, elapsed = generate_text(
    generator, 'the cat', max_new_tokens=30, temperature=1.0, top_p=0.9)
print(f'Top-p=0.9: {text!r}  ({len(ids)} tokens, {elapsed:.2f}s)')

### 6.5 Combined Parameters

All strategies can be combined: temperature + top-k + top-p.

In [ ]:
text, ids, elapsed = generate_text(
    generator, 'the cat', max_new_tokens=30,
    temperature=0.7, top_k=10, top_p=0.95)
print(f'Combined: {text!r}  ({len(ids)} tokens, {elapsed:.2f}s)')

---
## 7. Arabic Text Generation

Demonstrate generation with an Arabic prompt and RTL rendering.

In [ ]:
arabic_prompt = 'مرحبا'
try:
    text, ids, elapsed = generate_text(
        generator, arabic_prompt, max_new_tokens=20, temperature=0.8)
    print(f'Prompt: {arabic_prompt}')
    print(f'Output ({len(ids)} tokens, {elapsed:.2f}s):')
    render_text(text)
except Exception as exc:
    print(f'Arabic generation error: {exc}')

---
## 8. Error Handling

The demo app handles several error conditions gracefully:
- Empty prompt
- No checkpoint found
- Generation timeout (>60s)

Let's verify the empty-prompt guard:

In [ ]:
try:
    generate_text(generator, '', max_new_tokens=10)
    print('ERROR: Should have raised ValueError')
except ValueError as e:
    print(f'Caught expected error: {e}')

---
## 9. Running the Streamlit App

For the full interactive experience with sliders and a model selector dropdown,
run the Streamlit app from the project root:

```bash
cd SDAIA-gpt-project2026
streamlit run src/demo/app.py
```

The app provides:
- **Model selector**: dropdown to choose pretrained or fine-tuned checkpoint
- **Parameter sliders**: temperature, top-k, top-p, max new tokens
- **RTL rendering**: automatic right-to-left display for Arabic output
- **Generation timing**: shows elapsed time and token count